# Day 18 · Colab 2 — Multi-tool Agent Chaining + Redis Event Queue

**Agentic Systems Bootcamp**

Here the agent does real **orchestration**: it *chains* tools to place an order —
`check_inventory → create_order → send_confirmation` — mixing **synchronous** tools
(SQLite reads/writes) with an **asynchronous** one (enqueue an email job onto a Redis Stream,
handled later by a separate **worker** via a consumer group), plus **memory** for recent context and long-term recall.

Runs entirely in Colab: `fakeredis` provides `XADD` / `XREADGROUP` and Redis-style memory, SQLite is built in, and a tiny local vector store handles long-term recall.

> **Patterns (from the deck):** *routing* (pick the right tool), *prompt chaining* (feed one tool's
> output into the next), *parallel* reads, and the **sync-vs-async** boundary — slow side-effects go
> on a queue so the agent turn stays fast.

## Step 1 — Install & configure

In [1]:
!pip install -q anthropic fakeredis 2>/dev/null
import os, getpass
from dotenv import load_dotenv

load_dotenv()
ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
    # os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
if not os.environ.get('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('ANTHROPIC_API_KEY (blank = offline mock): ')
LIVE = bool(os.environ.get('ANTHROPIC_API_KEY'))
MODEL = 'claude-sonnet-4-6'
print('LIVE mode' if LIVE else 'OFFLINE mock mode')

LIVE mode


The system cannot find the path specified.


## Step 2 — SQLite orders + inventory database

Two tables: `inventory` (stock per SKU) and `orders` (what we create).
All access is through **parameterised** SQL — never string-formatted — and reads are row-limited.

In [2]:
import sqlite3
db = sqlite3.connect(':memory:', check_same_thread=False)
db.execute('CREATE TABLE inventory (sku TEXT PRIMARY KEY, name TEXT, qty INTEGER, price REAL)')
db.execute('CREATE TABLE orders (id INTEGER PRIMARY KEY AUTOINCREMENT, sku TEXT, qty INTEGER, total REAL, status TEXT)')
db.executemany('INSERT INTO inventory VALUES (?,?,?,?)', [
    ('KB-01', 'Mechanical keyboard', 12, 129.0),
    ('HUB-2', 'USB-C hub',           0,  58.0),
    ('MON-4', '4K monitor',          5, 410.0),
])
db.commit()
print('inventory seeded')

inventory seeded


## Step 3 — A Redis Stream as the email job queue

`send_confirmation` won't send email inline (slow, can fail). It **enqueues** a job with `XADD` and
returns a `job_id` immediately. A **worker** later reads with `XREADGROUP` (a consumer group gives us
at-least-once delivery + acks) and writes the result back to a `jobresult:<id>` key.

In [3]:
import fakeredis, json, time, uuid, math, re
r = fakeredis.FakeStrictRedis()
STREAM = 'emails'
GROUP  = 'mailers'
try:
    r.xgroup_create(STREAM, GROUP, id='0', mkstream=True)
except Exception as e:
    print('group exists:', e)

def enqueue_email(to, subject, body):
    job_id = uuid.uuid4().hex[:8]
    r.xadd(STREAM, {'job_id': job_id, 'to': to, 'subject': subject, 'body': body})
    r.set(f'jobresult:{job_id}', 'queued')
    return job_id

print('queue ready; sample job id ->', enqueue_email('a@x.com', 'hi', 'test'))

queue ready; sample job id -> 900b74c0


## Step 4 — The worker (consumer group)

In production this is a separate process. Here we drain the stream on demand. Each message is
processed, the result stored, and the message **acked** with `XACK` so it isn't redelivered.

In [4]:
def run_worker(max_msgs=10):
    processed = 0
    resp = r.xreadgroup(GROUP, 'worker-1', {STREAM: '>'}, count=max_msgs)
    for _stream, msgs in resp or []:
        for msg_id, fields in msgs:
            f = {k.decode(): v.decode() for k, v in fields.items()}
            # … here you'd call a real email provider …
            r.set(f"jobresult:{f['job_id']}", 'sent')
            r.xack(STREAM, GROUP, msg_id)
            processed += 1
    return processed

print('worker processed', run_worker(), 'job(s)')

worker processed 1 job(s)


## Step 5 ? Redis short-term memory + vector long-term recall

The queue Redis client can also hold short-term conversation memory. Recent turns go into a Redis list with `LPUSH`/`LTRIM`, so the agent gets bounded context instead of an ever-growing prompt.

Long-term recall goes into a tiny vector store. Each memory is embedded as a sparse hashed bag-of-words vector, then searched with cosine similarity. In production this same interface is where you would swap in Chroma, FAISS, pgvector, Redis Vector Sets, or a managed vector DB with model embeddings.


In [5]:
class RedisShortTermMemory:
    def __init__(self, redis_client, session_id='capstone-user', max_turns=8):
        self.r = redis_client
        self.key = f'hist:{session_id}'
        self.max_turns = max_turns

    def append(self, role, content):
        item = json.dumps({'role': role, 'content': content, 'ts': time.time()})
        self.r.rpush(self.key, item)
        self.r.ltrim(self.key, -self.max_turns, -1)

    def recent(self):
        rows = self.r.lrange(self.key, 0, -1)
        out = []
        for row in rows:
            if isinstance(row, bytes):
                row = row.decode()
            out.append(json.loads(row))
        return out

    def as_prompt(self):
        turns = self.recent()
        if not turns:
            return 'No recent turns.'
        return '\n'.join(f"{t['role']}: {t['content']}" for t in turns)


class TinyVectorStore:
    def __init__(self, dims=256):
        self.dims = dims
        self.docs = []

    def _tokens(self, text):
        return re.findall(r'[a-z0-9]+', text.lower())

    def _embed(self, text):
        vec = {}
        for tok in self._tokens(text):
            bucket = hash(tok) % self.dims
            vec[bucket] = vec.get(bucket, 0.0) + 1.0
        norm = math.sqrt(sum(v * v for v in vec.values())) or 1.0
        return {k: v / norm for k, v in vec.items()}

    def add(self, text, metadata=None):
        doc_id = uuid.uuid4().hex[:8]
        self.docs.append({'id': doc_id, 'text': text, 'metadata': metadata or {}, 'vector': self._embed(text)})
        return doc_id

    def search(self, query, k=3):
        q = self._embed(query)
        scored = []
        for doc in self.docs:
            score = sum(q.get(i, 0.0) * doc['vector'].get(i, 0.0) for i in q)
            if score > 0:
                scored.append({
                    'id': doc['id'],
                    'score': round(score, 3),
                    'text': doc['text'],
                    'metadata': doc['metadata'],
                })
        return sorted(scored, key=lambda x: x['score'], reverse=True)[:k]


short_memory = RedisShortTermMemory(r, session_id='capstone-user')
long_memory = TinyVectorStore()

# Seed durable memories the agent can retrieve later.
long_memory.add('Asha prefers express shipping and wants order confirmations by email.', {'kind': 'customer_fact'})
long_memory.add('4K monitor SKU MON-4 is commonly ordered by design teams.', {'kind': 'product_note'})


def remember_memory(text: str, kind: str = 'note'):
    doc_id = long_memory.add(text, {'kind': kind, 'ts': time.time()})
    return {'memory_id': doc_id, 'stored': True, 'kind': kind}


def recall_memory(query: str, k: int = 3):
    return {'query': query, 'matches': long_memory.search(query, k=k)}


def memory_context_for(user_text):
    recalled = recall_memory(user_text, k=3)['matches']
    recall_lines = [f"- ({m['score']}) {m['text']}" for m in recalled] or ['- No long-term matches.']
    return (
        'Recent Redis conversation memory:\n'
        f'{short_memory.as_prompt()}\n\n'
        'Long-term vector recall:\n'
        + '\n'.join(recall_lines)
    )

print('memory ready')
print(memory_context_for('Asha ordering a 4K monitor with email confirmation'))


memory ready
Recent Redis conversation memory:
No recent turns.

Long-term vector recall:
- (0.294) 4K monitor SKU MON-4 is commonly ordered by design teams.
- (0.224) Asha prefers express shipping and wants order confirmations by email.


## Step 5 — Tool functions (sync + async)

`check_inventory` and `create_order` are **synchronous** — they touch SQLite and return immediately.
`create_order` re-checks stock and decrements it in one transaction (guarding against overselling).
`send_confirmation` is **asynchronous** — it enqueues and returns a `job_id`. `check_job` polls a result.

In [6]:
def check_inventory(sku: str):
    row = db.execute('SELECT sku,name,qty,price FROM inventory WHERE sku=? LIMIT 1', (sku,)).fetchone()
    if not row:
        return {'error': f'unknown sku {sku}'}
    return {'sku': row[0], 'name': row[1], 'qty': row[2], 'price': row[3]}

def create_order(sku: str, qty: int):
    cur = db.execute('SELECT qty,price FROM inventory WHERE sku=? LIMIT 1', (sku,)).fetchone()
    if not cur:
        return {'error': f'unknown sku {sku}'}
    have, price = cur
    if qty <= 0:
        return {'error': 'qty must be positive'}
    if have < qty:
        return {'error': f'insufficient stock: have {have}, need {qty}'}
    db.execute('UPDATE inventory SET qty=qty-? WHERE sku=?', (qty, sku))
    cur2 = db.execute('INSERT INTO orders (sku,qty,total,status) VALUES (?,?,?,?)',
                      (sku, qty, round(price*qty, 2), 'created'))
    db.commit()
    return {'order_id': cur2.lastrowid, 'sku': sku, 'qty': qty, 'total': round(price*qty, 2)}

def send_confirmation(to: str, order_id: int):
    job_id = enqueue_email(to, f'Order {order_id} confirmed', f'Your order {order_id} is on its way.')
    return {'job_id': job_id, 'status': 'queued'}

def check_job(job_id: str):
    v = r.get(f'jobresult:{job_id}')
    return {'job_id': job_id, 'status': v.decode() if v else 'unknown'}

# quick manual chain
inv = check_inventory('KB-01'); print(inv)
od  = create_order('KB-01', 2);  print(od)
jb  = send_confirmation('asha@x.com', od['order_id']); print(jb)
run_worker(); print(check_job(jb['job_id']))

{'sku': 'KB-01', 'name': 'Mechanical keyboard', 'qty': 12, 'price': 129.0}
{'order_id': 1, 'sku': 'KB-01', 'qty': 2, 'total': 258.0}
{'job_id': '7d367537', 'status': 'queued'}
{'job_id': '7d367537', 'status': 'sent'}


## Step 6 — Tool schemas + dispatch

In [7]:
TOOLS = [
  {'name':'check_inventory','description':'Check stock and price for a product SKU before ordering.',
   'input_schema':{'type':'object','properties':{'sku':{'type':'string','description':'Product SKU, e.g. KB-01.'}},'required':['sku']}},
  {'name':'create_order','description':'Create an order for a SKU and quantity. Fails if stock is insufficient.',
   'input_schema':{'type':'object','properties':{'sku':{'type':'string'},'qty':{'type':'integer','description':'Units to order (>0).'}},'required':['sku','qty']}},
  {'name':'send_confirmation','description':'Queue a confirmation email for a created order. Returns a job_id immediately.',
   'input_schema':{'type':'object','properties':{'to':{'type':'string','description':'Customer email.'},'order_id':{'type':'integer'}},'required':['to','order_id']}},
  {'name':'check_job','description':'Check the status of a queued email job by job_id.',
   'input_schema':{'type':'object','properties':{'job_id':{'type':'string'}},'required':['job_id']}},
  {'name':'recall_memory','description':'Search long-term vector memory for facts relevant to the current request.',
   'input_schema':{'type':'object','properties':{'query':{'type':'string'},'k':{'type':'integer','description':'Number of matches to return.'}},'required':['query']}},
  {'name':'remember_memory','description':'Store a durable long-term memory after a useful fact or order outcome is learned.',
   'input_schema':{'type':'object','properties':{'text':{'type':'string'},'kind':{'type':'string','description':'fact, preference, order, or note.'}},'required':['text']}},
]
DISPATCH = {'check_inventory':check_inventory,'create_order':create_order,
            'send_confirmation':send_confirmation,'check_job':check_job,
            'recall_memory':recall_memory,'remember_memory':remember_memory}

def run_tool(name, args):
    fn = DISPATCH.get(name)
    if not fn: return {'error': f'unknown tool {name}'}, True
    try:
        out = fn(**args)
        return out, isinstance(out, dict) and 'error' in out
    except Exception as e:
        return {'error': repr(e)}, True
print('tools ready')


tools ready


## Step 7 — The orchestrating agent loop

Same `stop_reason` loop as Colab 1, but now the model **chains** several tools in one turn and we
drain the worker afterwards. The offline mock scripts the full chain so the notebook runs without a key.

In [8]:
import json
SYSTEM = (
  'You are an ordering agent. Use the supplied memory context before deciding what to do. '
  'To place an order: first recall_memory if the request depends on user/product history, then '
  'check_inventory, then create_order, then send_confirmation with the returned order_id. '
  'After a successful order, remember_memory with a concise order summary. '
  'Report the order total and the email job status. If stock is insufficient, say so and do not create the order.'
)

def agent(user_text, max_steps=8, verbose=True):
    context = memory_context_for(user_text)
    short_memory.append('user', user_text)

    if not LIVE:
        if verbose:
            print('... (mock) recall_memory -> check_inventory -> create_order -> send_confirmation -> remember_memory')
            print(context)
        recall,_ = run_tool('recall_memory', {'query': user_text, 'k': 2})
        inv,_ = run_tool('check_inventory', {'sku':'MON-4'})
        od,_  = run_tool('create_order', {'sku':'MON-4','qty':1})
        jb,_  = run_tool('send_confirmation', {'to':'asha@x.com','order_id':od['order_id']})
        run_worker()
        st,_  = run_tool('check_job', {'job_id':jb['job_id']})
        run_tool('remember_memory', {
            'text': f"Order {od['order_id']} for Asha: {od['qty']} x {od['sku']} total ${od['total']}; email {st['status']}.",
            'kind': 'order'
        })
        answer = f"(mock) Order {od['order_id']} total ${od['total']}; email {st['status']}. Recalled {len(recall['matches'])} memory match(es)."
        short_memory.append('assistant', answer)
        return answer

    from anthropic import Anthropic
    A = Anthropic()
    messages = [{'role':'user','content': f'{context}\n\nCurrent request: {user_text}'}]
    for _ in range(max_steps):
        resp = A.messages.create(model=MODEL, max_tokens=1024, system=SYSTEM, tools=TOOLS, messages=messages)
        if resp.stop_reason == 'tool_use':
            messages.append({'role':'assistant','content':[b.model_dump() for b in resp.content]})
            results = []
            for b in resp.content:
                if b.type == 'tool_use':
                    if verbose: print(f'  -> {b.name}({b.input})')
                    out, is_err = run_tool(b.name, b.input)
                    results.append({'type':'tool_result','tool_use_id':b.id,
                                    'content':json.dumps(out),'is_error':is_err})
            messages.append({'role':'user','content':results})
            run_worker()  # drain any queued email jobs between steps
            continue
        answer = ''.join(b.text for b in resp.content if b.type=='text')
        short_memory.append('assistant', answer)
        return answer
    answer = '(max steps reached)'
    short_memory.append('assistant', answer)
    return answer

print(agent('Order one 4K monitor (SKU MON-4) and email asha@x.com the confirmation.'))


  -> check_inventory({'sku': 'MON-4'})
  -> recall_memory({'query': 'Asha order history and preferences', 'k': 3})
  -> create_order({'sku': 'MON-4', 'qty': 1})
  -> send_confirmation({'to': 'asha@x.com', 'order_id': 2})
  -> remember_memory({'kind': 'order', 'text': 'Asha (asha@x.com) ordered 1x 4K Monitor (SKU MON-4) for $410.00. Order ID: 2. Confirmation sent to asha@x.com.'})
  -> check_job({'job_id': '84266f1e'})
Everything went through smoothly! Here's the full summary:

| Detail | Info |
|---|---|
| **Product** | 4K Monitor (SKU: MON-4) |
| **Quantity** | 1 |
| **Order Total** | $410.00 |
| **Order ID** | #2 |
| **Confirmation Email** | asha@x.com ✅ Sent |

> 📝 **Note:** Asha's preference for express shipping is on file — make sure the fulfillment team applies express shipping to Order #2!


## Step 8 — Show the resulting state

In [9]:
print('Orders table:')
for row in db.execute('SELECT id,sku,qty,total,status FROM orders').fetchall():
    print(' ', row)
print('Remaining stock:')
for row in db.execute('SELECT sku,qty FROM inventory').fetchall():
    print(' ', row)
print('Recent Redis memory:')
for turn in short_memory.recent():
    print(' ', turn['role'], '->', turn['content'])
print('Vector recall for Asha monitor:', recall_memory('Asha monitor order email', k=3))
# Try an out-of-stock order to see graceful failure
print('Out-of-stock attempt:', run_tool('create_order', {'sku':'HUB-2','qty':1}))


Orders table:
  (1, 'KB-01', 2, 258.0, 'created')
  (2, 'MON-4', 1, 410.0, 'created')
Remaining stock:
  ('KB-01', 10)
  ('HUB-2', 0)
  ('MON-4', 4)
Recent Redis memory:
  user -> Order one 4K monitor (SKU MON-4) and email asha@x.com the confirmation.
  assistant -> Everything went through smoothly! Here's the full summary:

| Detail | Info |
|---|---|
| **Product** | 4K Monitor (SKU: MON-4) |
| **Quantity** | 1 |
| **Order Total** | $410.00 |
| **Order ID** | #2 |
| **Confirmation Email** | asha@x.com ✅ Sent |

> 📝 **Note:** Asha's preference for express shipping is on file — make sure the fulfillment team applies express shipping to Order #2!
Vector recall for Asha monitor: {'query': 'Asha monitor order email', 'matches': [{'id': 'a19fd3b5', 'score': 0.474, 'text': 'Asha prefers express shipping and wants order confirmations by email.', 'metadata': {'kind': 'customer_fact'}}, {'id': '99cac114', 'score': 0.435, 'text': 'Asha (asha@x.com) ordered 1x 4K Monitor (SKU MON-4) for $410.00. 

## Extension tasks

1. **Dead-letter & retry.** Make the worker fail jobs whose `to` is invalid; move them to an `emails:dlq`
   stream after N retries instead of acking. Add a tool to inspect the DLQ.
2. **Evaluator / verify step.** Add a `verify_order` tool that re-reads the order and confirms total &
   status; have the system prompt require it before declaring success (evaluator-optimiser pattern).
3. **Orchestrator-worker split.** Move `run_worker()` into a background `threading.Thread` loop so the
   agent never calls it directly; the queue truly decouples them.
4. **Human-in-the-loop gate.** For orders above a value threshold, have `create_order` return
   `{'status':'needs_approval'}` and require an `approve_order(order_id)` tool call before fulfilment.
5. **Trace spans.** Wrap `run_tool` to record `{tool, args, ms, ok}` per call; print a per-turn trace table.
6. **Memory quality eval.** Add a small labeled set of user requests and measure whether vector recall returns the expected customer/order fact in the top 3.
7. **Prompt caching + token savings.** Mark the system prompt and tool block as cache-eligible and compare
   `usage.cache_read_input_tokens` across two identical turns; report the measured savings.

> **Stretch:** replace `fakeredis` with a real Redis (`redis.Redis.from_url(os.environ['REDIS_URL'])`) and
> run the worker as a genuinely separate Colab cell / process. The stream + consumer-group code is identical.